# ReelBench: Two-Tower + SASRec training (Kaggle/Colab GPU)

Kaggle/Colab GPU training run log for the two neural retrieval models in
[ReelBench](../README.md): the two-tower retriever and SASRec.

This replaces a corrupted notebook file that shipped in a previous commit
(invalid JSON, effectively unopenable). This notebook runs the exact same
`src.models.two_tower` / `src.models.sasrec` training and export code the
CLI scripts (`scripts/train_two_tower.py`, `scripts/train_sasrec.py`) use,
so results here match a CLI run with the same arguments.

**Before running:** upload `data/processed/train.parquet` (written by
`scripts/run_phase1.py`) to this notebook's working directory, or point
`TRAIN_PATH` below at wherever you've placed it (a mounted Google Drive
path on Colab, `/kaggle/input/...` on Kaggle).

**Runtime:** GPU (both models fall back to CPU automatically if none is
available, but training is designed for a free-tier GPU session).

## 1. Setup

In [1]:
import os

REPO_URL = "https://github.com/abhinavharbola/reelbench.git"
REPO_DIR = "reelbench"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")


Cloning into 'reelbench'...


In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path(REPO_DIR) if Path(REPO_DIR).exists() else Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

TRAIN_PATH = Path("/kaggle/input/datasets/abhinavharbola/reelbench-train-set/train.parquet")
OUTPUT_DIR = REPO_ROOT / "data/processed"

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle").exists() else Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"train path:      {TRAIN_PATH}")
print(f"output dir:       {OUTPUT_DIR}")
print(f"checkpoint dir:   {CHECKPOINT_DIR}")


train path:      /kaggle/input/datasets/abhinavharbola/reelbench-train-set/train.parquet
output dir:       reelbench/data/processed
checkpoint dir:   /kaggle/working/checkpoints


In [3]:
!pip install -q polars pyarrow mlflow
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 11.8 MB/s eta 0:00:00


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


CUDA available: True
device: Tesla T4


In [5]:
import polars as pl

train_df = pl.read_parquet(TRAIN_PATH)
print(f"{train_df.height:,} interactions, {train_df['userId'].n_unique():,} users, "
      f"{train_df['movieId'].n_unique():,} items")
train_df.head()


11,202,874 interactions, 148,745 users, 31,195 items


userId,movieId,timestamp
i32,i32,i64
1,5952,1147868053
1,1653,1147868097
1,1250,1147868414
1,6377,1147868469
1,3448,1147868480


## 2. Two-tower

In-batch negative sampling, checkpointed every epoch (see
`src/models/two_tower.py`). Re-running this cell after a disconnect
resumes automatically from the last completed epoch found at
`CHECKPOINT_DIR / "two_tower.pt"`.

In [6]:
from src.models.two_tower import export_embeddings as export_two_tower_embeddings
from src.models.two_tower import train as train_two_tower

two_tower_checkpoint = CHECKPOINT_DIR / "two_tower.pt"

two_tower_model, two_tower_id_maps = train_two_tower(
    train_df,
    checkpoint_path=two_tower_checkpoint,
    epochs=10,
    batch_size=512,
    lr=1e-3,
    embedding_dim=64,
)


epoch 0: avg_loss=6.1409
epoch 1: avg_loss=5.8062
epoch 2: avg_loss=5.6325
epoch 3: avg_loss=5.5653
epoch 4: avg_loss=5.5100
epoch 5: avg_loss=5.4683
epoch 6: avg_loss=5.4248
epoch 7: avg_loss=5.3895
epoch 8: avg_loss=5.3633
epoch 9: avg_loss=5.3378


2026/09/14 10:54:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/14 10:54:12 INFO mlflow.store.db.utils: Updating database tables
2026/09/14 10:54:14 INFO mlflow.tracking.fluent: Experiment with name 'movielens-recsys-benchmark' does not exist. Creating a new experiment.


In [7]:
export_two_tower_embeddings(two_tower_model, two_tower_id_maps, OUTPUT_DIR)


exported 148745 user and 31195 item embeddings to reelbench/data/processed


## 3. SASRec

Causal self-attention over each user's chronological sequence, next-item
prediction (see `src/models/sasrec.py`). Same checkpoint-resume pattern as
above, keyed on `CHECKPOINT_DIR / "sasrec.pt"`.

In [8]:
from src.models.sasrec import build_user_sequences
from src.models.sasrec import export_embeddings as export_sasrec_embeddings
from src.models.sasrec import train as train_sasrec

sasrec_checkpoint = CHECKPOINT_DIR / "sasrec.pt"

sasrec_model, sasrec_id_maps, sasrec_config = train_sasrec(
    train_df,
    checkpoint_path=sasrec_checkpoint,
    epochs=10,
    batch_size=128,
    lr=1e-3,
    max_seq_len=50,
    embedding_dim=64,
)


epoch 0: avg_loss=9.8021
epoch 1: avg_loss=7.6704
epoch 2: avg_loss=7.4614
epoch 3: avg_loss=7.2235
epoch 4: avg_loss=7.0616
epoch 5: avg_loss=6.9661
epoch 6: avg_loss=6.8594
epoch 7: avg_loss=6.7727
epoch 8: avg_loss=6.7108
epoch 9: avg_loss=6.6540


In [9]:
# Use sasrec_config, not the literal args above: if this run resumed from a
# checkpoint trained with a different max_seq_len, train() silently used the
# checkpoint's value internally, and exporting with a mismatched value here
# would crash on a position_embedding shape mismatch. See train()'s
# docstring in src/models/sasrec.py.
sequences = build_user_sequences(train_df, sasrec_id_maps)
export_sasrec_embeddings(
    sasrec_model, sasrec_id_maps, sequences, OUTPUT_DIR,
    max_seq_len=sasrec_config["max_seq_len"],
)


exported 148745 user and 31195 item embeddings to reelbench/data/processed


## 4. Sanity-check the exported embeddings

Confirms neither export produced NaNs before you copy the parquet files
back to your local `data/processed/` (see `scripts/check_embeddings_for_nan.py`
for the same check from the command line).

In [10]:
import numpy as np

for prefix in ["two_tower", "sasrec"]:
    for kind in ["user", "item"]:
        path = OUTPUT_DIR / f"{prefix}_{kind}_embeddings.parquet"
        df = pl.read_parquet(path)
        arr = np.array(df["embedding"].to_list())
        n_nan = int(np.isnan(arr).any(axis=1).sum())
        print(f"{path.name}: {arr.shape[0]} rows, dim {arr.shape[1]}, {n_nan} rows with NaN")


two_tower_user_embeddings.parquet: 148745 rows, dim 64, 0 rows with NaN
two_tower_item_embeddings.parquet: 31195 rows, dim 64, 0 rows with NaN
sasrec_user_embeddings.parquet: 148745 rows, dim 64, 0 rows with NaN
sasrec_item_embeddings.parquet: 31195 rows, dim 64, 0 rows with NaN


## 5. Next steps

Download `data/processed/two_tower_{user,item}_embeddings.parquet` and
`data/processed/sasrec_{user,item}_embeddings.parquet` from this session,
copy them into your local `data/processed/`, then locally run:

```bash
python scripts/build_serving_artifacts.py
python scripts/build_ui_artifacts.py
python scripts/evaluate_pipeline_models.py
```